# Session Tracker

Run this notebook at the end of your coding session to auto-log what you modified today.

In [1]:
import os
import json
from datetime import datetime, timedelta
import pathlib

# Get workspace root (parent of logs folder)
workspace_root = pathlib.Path.cwd().parent
logs_dir = workspace_root / 'logs'

# Today's date
today = datetime.now().strftime('%Y-%m-%d')
print(f'Scanning for notebooks modified on {today}...')
print(f'Workspace root: {workspace_root}')

Scanning for notebooks modified on 2026-04-22...
Workspace root: z:\Lauryn\Liposarcoma\Liposarcoma_2025\LPS_863_Abema_July_2025\single_cell_analysis


In [2]:
# Scan for notebooks modified today
modified_notebooks = []
modified_scripts = []

# Walk through workspace excluding .git, __pycache__, .conda, packages, python_environments
exclude_dirs = {'.git', '__pycache__', '.conda', '.ipynb_checkpoints', 'packages', 'python_environments', 'environment'}

for root, dirs, files in os.walk(workspace_root):
    # Remove excluded directories from dirs in-place to prevent walking into them
    dirs[:] = [d for d in dirs if d not in exclude_dirs]
    
    for file in files:
        if file.startswith('.'):
            continue
            
        file_path = pathlib.Path(root) / file
        
        # Get modification time
        mtime = datetime.fromtimestamp(os.path.getmtime(file_path))
        mtime_str = mtime.strftime('%Y-%m-%d')
        
        # Check if modified today
        if mtime_str == today:
            rel_path = file_path.relative_to(workspace_root)
            mod_time_str = mtime.strftime('%H:%M')
            
            if file.endswith('.ipynb'):
                modified_notebooks.append({
                    'path': str(rel_path),
                    'timestamp': mod_time_str,
                    'datetime': mtime
                })
            elif file.endswith('.py'):
                modified_scripts.append({
                    'path': str(rel_path),
                    'timestamp': mod_time_str,
                    'datetime': mtime
                })

# Sort by modification time
modified_notebooks.sort(key=lambda x: x['datetime'], reverse=True)
modified_scripts.sort(key=lambda x: x['datetime'], reverse=True)

print(f'\nFound {len(modified_notebooks)} notebooks modified today')
print(f'Found {len(modified_scripts)} scripts modified today')
print('\nNotebooks:')
for nb in modified_notebooks:
    print(f"  {nb['timestamp']} - {nb['path']}")
print('\nScripts:')
for script in modified_scripts:
    print(f"  {script['timestamp']} - {script['path']}")


Found 36 notebooks modified today
Found 5 scripts modified today

Notebooks:
  15:52 - 00_experiment_setup\experiment_details_LPS863_Abema_July2025.ipynb
  15:41 - analysis_tools\liposarcoma\00_adata_conversion_pRB_ratio_CD45_filtering.ipynb
  15:41 - analysis_tools\liposarcoma\00_adata_conversion_pRB_ratio_lipo.ipynb
  15:40 - analysis_tools\slingshot\Features_Plotted_by_Lineage_Pseduotime_from_Slingshot.ipynb
  15:40 - analysis_tools\slingshot\3D_Phate_plotting_with_features and_Lineage_from_Slingshot.ipynb
  15:40 - analysis_tools\slingshot\3D_Phate_plotting_with_Lineage_from_Slingshot.ipynb
  15:40 - analysis_tools\liposarcoma\treatment_condition_histograms.ipynb
  15:40 - analysis_tools\liposarcoma\sas_treatment_separation_lipo.ipynb
  15:40 - analysis_tools\liposarcoma\pRB_RB_ratio_separation_lipo.ipynb
  15:40 - analysis_tools\liposarcoma\incorporating_pRB_ratio_adata_lipo.ipynb
  15:40 - analysis_tools\liposarcoma\fold_change_graph.ipynb
  15:40 - analysis_tools\liposarcoma\Vi

In [3]:
# Generate markdown log entry
log_entry = f"""# Daily Log: {today}

**Date:** {today}  
**Auto-generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  

---

## Notebooks Modified

"""

if modified_notebooks:
    for nb in modified_notebooks:
        log_entry += f"- `{nb['path']}` ({nb['timestamp']})\n"
else:
    log_entry += "*None*\n"

log_entry += f"""
## Scripts Modified

"""

if modified_scripts:
    for script in modified_scripts:
        log_entry += f"- `{script['path']}` ({script['timestamp']})\n"
else:
    log_entry += "*None*\n"

log_entry += f"""
## Summary

**Total items modified:** {len(modified_notebooks) + len(modified_scripts)}

### Status
- [ ] Pipeline steps completed
- [ ] Analysis notebooks run
- [ ] Results saved

### Notes
Add any observations or next steps below:

"""

print(log_entry)

# Daily Log: 2026-04-22

**Date:** 2026-04-22  
**Auto-generated:** 2026-04-22 16:19:47  

---

## Notebooks Modified

- `00_experiment_setup\experiment_details_LPS863_Abema_July2025.ipynb` (15:52)
- `analysis_tools\liposarcoma\00_adata_conversion_pRB_ratio_CD45_filtering.ipynb` (15:41)
- `analysis_tools\liposarcoma\00_adata_conversion_pRB_ratio_lipo.ipynb` (15:41)
- `analysis_tools\slingshot\Features_Plotted_by_Lineage_Pseduotime_from_Slingshot.ipynb` (15:40)
- `analysis_tools\slingshot\3D_Phate_plotting_with_features and_Lineage_from_Slingshot.ipynb` (15:40)
- `analysis_tools\slingshot\3D_Phate_plotting_with_Lineage_from_Slingshot.ipynb` (15:40)
- `analysis_tools\liposarcoma\treatment_condition_histograms.ipynb` (15:40)
- `analysis_tools\liposarcoma\sas_treatment_separation_lipo.ipynb` (15:40)
- `analysis_tools\liposarcoma\pRB_RB_ratio_separation_lipo.ipynb` (15:40)
- `analysis_tools\liposarcoma\incorporating_pRB_ratio_adata_lipo.ipynb` (15:40)
- `analysis_tools\liposarcoma\fold_chan

In [4]:
# Save log entry
log_filename = logs_dir / f"{today}_session_log.md"

# Check if log already exists
if log_filename.exists():
    print(f"Log already exists: {log_filename}")
    print("Appending to existing log...\n")
    with open(log_filename, 'a') as f:
        f.write("\n---\n\n## Updated at " + datetime.now().strftime('%H:%M:%S') + "\n\n")
        f.write(f"Notebooks modified: {len(modified_notebooks)} | Scripts modified: {len(modified_scripts)}\n")
        for nb in modified_notebooks:
            f.write(f"- {nb['path']}\n")
        for script in modified_scripts:
            f.write(f"- {script['path']}\n")
else:
    print(f"Creating new log: {log_filename}\n")
    with open(log_filename, 'w') as f:
        f.write(log_entry)

print(f"✅ Log saved to: {log_filename}")
print(f"\nOpen it to add notes: {log_filename}")

Creating new log: z:\Lauryn\Liposarcoma\Liposarcoma_2025\LPS_863_Abema_July_2025\single_cell_analysis\logs\2026-04-22_session_log.md

✅ Log saved to: z:\Lauryn\Liposarcoma\Liposarcoma_2025\LPS_863_Abema_July_2025\single_cell_analysis\logs\2026-04-22_session_log.md

Open it to add notes: z:\Lauryn\Liposarcoma\Liposarcoma_2025\LPS_863_Abema_July_2025\single_cell_analysis\logs\2026-04-22_session_log.md
